# 01 — Model Known Base Multi-Class + Statistik Per-Kelas (Paper 3, Tahap T1)

**Dijalankan di SageMaker.** Fondasi Paper 3 (class-incremental / open-set NIDS).
Melanjutkan `../../unswnb-15/notebooks/24_multiclass.ipynb` (JANGAN mengulang):
loader 9-fitur SFM, `map_cic`/`map_uns`, dan konfigurasi XGBoost `multi:softprob`
dipakai ulang apa adanya.

**Yang BARU di notebook ini (khusus Paper 3):**
1. **Split kelas KNOWN vs HELD-OUT.** Model hanya dilatih pada kelas *known*; kelas
   *held-out* disimpan sebagai "serangan baru" untuk menguji open-set di notebook 02.
2. **Simpan artefak deployment multi-class**: model + scaler + `LabelEncoder`.
3. **Simpan CENTROID + KOVARIANS per-kelas** (di ruang ter-scale) sebagai bekal skor
   **Mahalanobis** untuk open-set recognition (notebook 02).

**Fitur:** 9 SFM Model A `duration, fwd_pkts, bwd_pkts, fwd_bytes, bwd_bytes, fwd_mean,
bwd_mean, src_load, dst_load`. Audit satuan: `duration` = MIKRODETIK (CIC ×1; UNSW dur×1e6).

**Output** (→ S3 `evolusion/known_base/`):
`known_base_results.json`, `model_known_<DS>.json`, `deploy_meta_mc_<DS>.json`
(scaler + centroid + inv-cov per-kelas + label map), confusion PNG, per-class CSV.

> Prinsip kejujuran: seluruh angka dari eksekusi nyata; kelas < MIN_CLASS digabung
> `Other-rare` agar stabil; scaler & statistik di-fit HANYA pada train (no leakage).

In [ ]:
import importlib.util as u, sys, subprocess
need=[m for m in ('xgboost','scikit-learn','scipy','pandas','numpy','matplotlib','boto3') if u.find_spec(m.replace('scikit-learn','sklearn')) is None]
if need: subprocess.run([sys.executable,'-m','pip','install','-q',*need],check=True)
print('setup ok' if not need else f'installed {need}')
print('=== SEL 0 (setup) SELESAI ===')

In [ ]:
import os, json, glob, datetime
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import (confusion_matrix, classification_report,
    matthews_corrcoef, f1_score, balanced_accuracy_score)
from sklearn.preprocessing import StandardScaler, LabelEncoder
import xgboost as xgb
plt.rcParams.update({'figure.dpi':120,'font.size':9})
S3_BUCKET=os.environ.get('S3_BUCKET','ssh-detection-features-232032302717')
S3_PREFIX='evolusion'; REGION=os.environ.get('AWS_REGION','ap-southeast-1')
OUTDIR='known_base_out'; os.makedirs(OUTDIR,exist_ok=True)
CANON=['duration','fwd_pkts','bwd_pkts','fwd_bytes','bwd_bytes','fwd_mean','bwd_mean','src_load','dst_load']
SEED=42; MIN_CLASS=200  # kelas < MIN_CLASS digabung 'Other-rare' agar stabil

# --- Skema KNOWN vs HELD-OUT (keputusan Paper 3; boleh diubah, catat di documentation.md) ---
# Held-out = jenis 'serangan baru' yang DISEMBUNYIKAN saat latih -> diuji open-set di nb 02.
# Default awal (dapat direvisi setelah lihat value_counts nyata di SEL 2):
HELDOUT = {
  'CIC':  ['Botnet', 'Infiltration'],          # known: Benign, BruteForce, DoS, DDoS, Web
  'UNSW': ['Worms', 'Shellcode', 'Backdoor'],  # known: Benign, Generic, Exploits, Fuzzers, DoS, Recon, Analysis
}
RESULTS={'generated':datetime.datetime.utcnow().isoformat()+'Z','features':CANON,'seed':SEED,
         'min_class':MIN_CLASS,'heldout':HELDOUT}
def savefig(n): p=os.path.join(OUTDIR,n); plt.savefig(p,bbox_inches='tight'); plt.close(); print(' saved',p); return p
def first(paths):
    for p in paths:
        h=sorted(glob.glob(p))
        if h: return h[0]
    return None
print('=== SEL 1 (config) SELESAI ===')

## 2. Muat data + pemetaan label → kategori (dipakai ulang dari nb 24)

In [ ]:
def map_cic(lbl):
    s=str(lbl).strip().lower()
    if s in ('benign','normal'): return 'Benign'
    if s.startswith('ddos') or 'loic' in s or 'hoic' in s: return 'DDoS'
    if s.startswith('dos'): return 'DoS'
    if 'bruteforce' in s or 'brute force' in s or 'ftp-brute' in s or 'ssh-brute' in s: return 'BruteForce'
    if s=='bot' or 'botnet' in s: return 'Botnet'
    if 'infil' in s: return 'Infiltration'
    if 'web' in s or 'xss' in s or 'sql' in s: return 'Web'
    return 'Other'
def map_uns(lbl):
    s=str(lbl).strip().lower()
    if s in ('normal','benign',''): return 'Benign'
    return {'dos':'DoS','exploits':'Exploits','fuzzers':'Fuzzers','generic':'Generic',
            'reconnaissance':'Recon','backdoor':'Backdoor','backdoors':'Backdoor',
            'shellcode':'Shellcode','worms':'Worms','analysis':'Analysis'}.get(s,'Other')

def load_cic():
    p=first(['../../CICDDoS2018/data/file_100.csv','../../CICDDoS2018/data/file_*.csv'])
    if not p: print('CIC csv tak ada'); return None
    c=pd.read_csv(p, low_memory=False); c.columns=c.columns.str.strip()
    cm={'duration':'Flow Duration','fwd_pkts':'Tot Fwd Pkts','bwd_pkts':'Tot Bwd Pkts','fwd_bytes':'TotLen Fwd Pkts',
        'bwd_bytes':'TotLen Bwd Pkts','fwd_mean':'Fwd Pkt Len Mean','bwd_mean':'Bwd Pkt Len Mean','src_load':'Flow Byts/s','dst_load':'Bwd Pkts/s'}
    lc=[x for x in c.columns if x.lower()=='label']; LAB=lc[0] if lc else c.columns[-1]
    if not all(v in c.columns for v in cm.values()): print('CIC kolom kurang'); return None
    d=pd.DataFrame({k:pd.to_numeric(c[cm[k]],errors='coerce') for k in CANON})
    d['cat']=c[LAB].map(map_cic)
    d=d.replace([np.inf,-np.inf],np.nan).dropna(); d=d[d['cat']!='Other']
    return d

def _pick_unsw_train():
    # Isi berkas UNSW TERTUKAR dgn namanya (Paper 1 §9.3): 'training-set.csv'=82k, 'testing-set.csv'=175k.
    # Pilih berdasarkan JUMLAH RECORD TERBANYAK (=175k) sebagai data latih, BUKAN dari nama berkas.
    cands=[]
    for pat in ['../data/UNSW_NB15_*set.csv','../../unswnb-15/data/UNSW_NB15_*set.csv']:
        cands+=sorted(glob.glob(pat))
    cands=list(dict.fromkeys(cands))  # unik, jaga urutan
    if not cands: return None
    best,best_n=None,-1
    for p in cands:
        try: n=sum(1 for _ in open(p,'r',errors='ignore'))-1  # jumlah baris data (minus header)
        except Exception: n=-1
        print(f'    kandidat UNSW: {os.path.basename(p)} ~{n} record')
        if n>best_n: best,best_n=p,n
    print(f'    -> pilih data latih (terbanyak): {os.path.basename(best)} (~{best_n} record)')
    return best

def load_uns():
    p=_pick_unsw_train()
    if not p: print('UNSW csv tak ada'); return None
    u2=pd.read_csv(p); need=['dur','spkts','dpkts','sbytes','dbytes','smean','dmean','sload','dload','attack_cat']
    if not all(x in u2.columns for x in need): print('UNSW kolom kurang'); return None
    d=pd.DataFrame({'duration':pd.to_numeric(u2['dur'],errors='coerce')*1e6,'fwd_pkts':u2['spkts'],'bwd_pkts':u2['dpkts'],
                    'fwd_bytes':u2['sbytes'],'bwd_bytes':u2['dbytes'],'fwd_mean':u2['smean'],'bwd_mean':u2['dmean'],
                    'src_load':pd.to_numeric(u2['sload'],errors='coerce')/8.0,
                    'dst_load':pd.to_numeric(u2['dpkts'],errors='coerce')/pd.to_numeric(u2['dur'],errors='coerce').replace(0,np.nan)})
    d['cat']=u2['attack_cat'].fillna('Normal').map(map_uns)
    d=d.replace([np.inf,-np.inf],np.nan).dropna(); d=d[d['cat']!='Other']
    return d

cic=load_cic(); uns=load_uns()
for nm,df in [('CIC',cic),('UNSW',uns)]:
    if df is not None: print(nm,'distribusi kelas:',df['cat'].value_counts().to_dict())
print('>> Periksa distribusi di atas, sesuaikan HELDOUT (SEL 1) bila perlu, lalu jalankan ulang dari SEL 3.')
print('=== SEL 2 (muat + peta) SELESAI ===')

## 3. Latih model KNOWN + hitung centroid & inv-kovarians per-kelas (bekal open-set)

In [ ]:
def merge_rare(df, min_n=MIN_CLASS):
    vc=df['cat'].value_counts(); rare=[c for c,n in vc.items() if n<min_n and c!='Benign']
    if rare:
        df=df.copy(); df['cat']=df['cat'].where(~df['cat'].isin(rare),'Other-rare')
    return df

def class_stats(Xtr_scaled, ytr, labels, reg=1e-6):
    """Centroid (mu_k) + inverse covariance (Sigma_k^{-1}) per-kelas di ruang ter-scale.
    Bekal skor Mahalanobis open-set di nb 02. reg = regularisasi diagonal agar invertibel."""
    stats={}
    d=Xtr_scaled.shape[1]
    for ci,cname in enumerate(labels):
        Xc=Xtr_scaled[ytr==ci]
        if len(Xc)<d+1:  # sampel < dimensi -> kovarians tak stabil; pakai identitas
            mu=Xc.mean(0) if len(Xc) else np.zeros(d); cov=np.eye(d)
        else:
            mu=Xc.mean(0); cov=np.cov(Xc, rowvar=False)+reg*np.eye(d)
        stats[cname]={'mu':mu.tolist(),'inv_cov':np.linalg.pinv(cov).tolist(),'n':int(len(Xc))}
    return stats

def train_known(df, name):
    if df is None or len(df)==0: return None
    heldout=HELDOUT.get(name,[])
    df_known=merge_rare(df[~df['cat'].isin(heldout)].copy())
    df_held =df[df['cat'].isin(heldout)].copy()
    print(f"[{name}] known={sorted(df_known['cat'].unique())} | held-out={sorted(df_held['cat'].unique())} "
          f"(n_known={len(df_known)}, n_held={len(df_held)})")
    X=df_known[CANON].values; le=LabelEncoder(); y=le.fit_transform(df_known['cat'].values)
    Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=0.3,random_state=SEED,stratify=y)
    sc=StandardScaler().fit(Xtr)  # FIT hanya train (no leakage)
    Xtr_s=sc.transform(Xtr); Xte_s=sc.transform(Xte)
    clf=xgb.XGBClassifier(objective='multi:softprob',num_class=len(le.classes_),
        max_depth=8,learning_rate=0.1,n_estimators=200,subsample=0.8,colsample_bytree=0.8,
        tree_method='hist',eval_metric='mlogloss',random_state=SEED,n_jobs=-1)
    clf.fit(Xtr_s,ytr)
    yp=np.asarray(clf.predict(Xte_s)).ravel().astype(int)
    labels=list(le.classes_)
    cm=confusion_matrix(yte,yp)
    rep=classification_report(yte,yp,target_names=labels,output_dict=True,zero_division=0)
    stats=class_stats(Xtr_s,ytr,labels)  # centroid + inv-cov per-kelas (train saja)
    res={'dataset':name,'known_classes':labels,'heldout_classes':sorted(df_held['cat'].unique().tolist()),
         'n_train':int(len(Xtr)),'n_test':int(len(yte)),
         'macro_f1':round(float(f1_score(yte,yp,average='macro')),4),
         'weighted_f1':round(float(f1_score(yte,yp,average='weighted')),4),
         'mcc':round(float(matthews_corrcoef(yte,yp)),4),
         'balanced_acc':round(float(balanced_accuracy_score(yte,yp)),4),
         'per_class':{c:{'precision':round(rep[c]['precision'],4),'recall':round(rep[c]['recall'],4),
                         'f1':round(rep[c]['f1-score'],4),'support':int(rep[c]['support'])} for c in labels},
         'confusion':cm.tolist()}
    print(f"[{name}] KNOWN macro-F1={res['macro_f1']} MCC={res['mcc']} bal-acc={res['balanced_acc']}")
    # --- simpan artefak deployment multi-class + statistik open-set ---
    clf.get_booster().save_model(os.path.join(OUTDIR,f'model_known_{name}.json'))
    meta={'features':CANON,'labels':labels,
          'scaler_mean':sc.mean_.tolist(),'scaler_scale':sc.scale_.tolist(),
          'class_stats':stats}
    json.dump(meta,open(os.path.join(OUTDIR,f'deploy_meta_mc_{name}.json'),'w'))
    print(f"    tersimpan model_known_{name}.json + deploy_meta_mc_{name}.json (mu/inv_cov {len(stats)} kelas)")
    return res,cm,labels

def plot_cm(cm, labels, title, fname):
    M=np.array(cm,float); row=M.sum(1,keepdims=True); row[row==0]=1; M=M/row
    fig,ax=plt.subplots(figsize=(1.3+0.7*len(labels),1.1+0.7*len(labels)))
    im=ax.imshow(M,cmap='Blues',vmin=0,vmax=1)
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels,rotation=45,ha='right')
    ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            ax.text(j,i,f'{M[i,j]:.2f}',ha='center',va='center',fontsize=7,color='white' if M[i,j]>0.5 else 'black')
    ax.set_title(title); fig.colorbar(im,ax=ax,fraction=0.046); plt.tight_layout(); savefig(fname)
print('=== SEL 3 (fungsi latih known + statistik) SELESAI ===')

## 4. Jalankan: latih model known CIC & UNSW + simpan artefak

In [ ]:
RESULTS['known_base']={}
for name,df in [('CIC',cic),('UNSW',uns)]:
    out=train_known(df,name)
    if out is None: continue
    res,cm,labels=out; RESULTS['known_base'][name]=res
    plot_cm(cm,labels,f'{name} KNOWN base (recall)',f'confusion_known_{name}.png')
    import IPython.display as ipd
    print(f'\n=== {name} per-class (known) ==='); ipd.display(pd.DataFrame(res['per_class']).T[['precision','recall','f1','support']])
print('=== SEL 4 (latih known CIC & UNSW) SELESAI ===')

## 5. Ringkas + simpan + UPLOAD S3

In [ ]:
rows=[]
for nm,r in RESULTS.get('known_base',{}).items():
    rows.append({'dataset':nm,'n_known':len(r['known_classes']),'held_out':','.join(r['heldout_classes']),
                 'macro_F1':r['macro_f1'],'weighted_F1':r['weighted_f1'],'MCC':r['mcc'],'bal_acc':r['balanced_acc']})
summ=pd.DataFrame(rows)
import IPython.display as ipd; print('Ringkasan model KNOWN base:'); ipd.display(summ)
summ.to_csv(os.path.join(OUTDIR,'known_base_summary.csv'),index=False)
jp=os.path.join(OUTDIR,'known_base_results.json'); json.dump(RESULTS,open(jp,'w'),indent=2); print('tersimpan',jp)
try:
    import boto3; s3=boto3.client('s3',region_name=REGION); up=0
    for fn in sorted(os.listdir(OUTDIR)):
        if fn.endswith(('.json','.png','.csv')): s3.upload_file(os.path.join(OUTDIR,fn),S3_BUCKET,f'{S3_PREFIX}/known_base/{fn}'); up+=1
    print(f'upload {up} artefak -> s3://{S3_BUCKET}/{S3_PREFIX}/known_base/')
except Exception as e: print('upload gagal:',e)
print('=== SEL 5 (simpan + upload) SELESAI ===')
print('SELESAI T1. Artefak deploy_meta_mc_*.json (mu + inv_cov per-kelas) siap dipakai nb 02 (open-set Mahalanobis).')